# 01 · numpy：一切的地基

**时间预算**：3 小时。前 2 小时跟着跑代码并随手改数字玩，最后 1 小时做第 10 节的练习。

学完这节课你应该能回答三个问题：

1. 已经有 `list` 了，为什么还要 numpy 数组？
2. `axis=0` 和 `axis=1` 到底是怎么算的？
3. 广播机制是什么，为什么它能让代码短十倍？

> 别急着往下翻。先自己想 30 秒第一个问题，再往下看证据。

## 1. 为什么不用 list？

先看证据，再看道理。下面这段代码对 100 万个数做同一件事：乘 2 加 1。

In [ ]:
import timeit

import numpy as np

n = 1_000_000
py_list = list(range(n))
np_arr = np.arange(n)

# timeit.repeat 跑 5 次取最小值，比手动掐表可靠：最小值最接近「没有干扰」的真实速度
t_list = min(timeit.repeat('[x * 2 + 1 for x in py_list]', globals=globals(), number=1, repeat=5))
t_np = min(timeit.repeat('np_arr * 2 + 1', globals=globals(), number=1, repeat=5))

print(f'数据量       : {n:,}')
print(f'list 推导式  : {t_list * 1000:8.2f} 毫秒')
print(f'numpy 向量化 : {t_np * 1000:8.2f} 毫秒')
print(f'快了         : {t_list / t_np:8.0f} 倍')

再看内存。list 里存的**不是数字本身，而是指向数字对象的指针**。

In [ ]:
import sys

# 注意：sys.getsizeof(py_list) 只算了指针数组，还要加上每个整数对象本身
list_total = sys.getsizeof(py_list) + sum(map(sys.getsizeof, py_list))
np_total = np_arr.nbytes

print(f'list 里一个整数对象         : {sys.getsizeof(py_list[0])} 字节')
print(f'list 总占用（指针 + 每个整数对象）: {list_total / 1024 / 1024:6.1f} MB')
print(f'numpy 总占用（连续 8 字节整数）   : {np_total / 1024 / 1024:6.1f} MB')
print(f'内存省了                    : {list_total / np_total:6.1f} 倍')
print(f'numpy 元素类型 {np_arr.dtype}，每个元素 {np_arr.itemsize} 字节')

### 结论：numpy 快，是因为三件事

1. **同类型**——整个数组只有一种 dtype，每个元素占固定字节数，位置可以算出来；
2. **连续内存**——数据挨着放，CPU 缓存几乎全命中；
3. **循环在 C 里跑**——`np_arr * 2 + 1` 这句话没有进入 Python 解释器逐元素循环，
   而是交给编译好的 C 代码批量处理。

这三条合起来叫**向量化**（vectorization）。把「逐元素循环」换成「整体操作」，
是从写 Python 到写数值计算代码最关键的一次思维转变。

记住这个词——后面 PyTorch 的 Tensor 是同一个思路，GPU 把它推得更极端。

## 2. 造数组的几种方式

下面每个都要跑一遍，重点记 `arange` 和 `linspace` 的区别。

In [ ]:
print('全 0  :', np.zeros(5))
print('全 1  :\n', np.ones((2, 3)))
print('全 7  :\n', np.full((2, 2), 7))
print('单位阵:\n', np.eye(3))

# arange：起点、终点（不含）、步长
print('arange(0, 10, 2) :', np.arange(0, 10, 2))
# linspace：起点、终点（含）、个数
print('linspace(0, 1, 5):', np.linspace(0, 1, 5))

rng = np.random.default_rng(42)  # 固定种子 = 每次结果一样，方便复现
print('正态随机 5 个 :', rng.normal(0, 1, 5).round(3))
print('整数随机 10 个:', rng.integers(1, 7, 10))  # 掷骰子，注意不含 7

> **新手最常混的两个函数**：`arange` 给的是**步长**、终点不含；`linspace` 给的是**个数**、终点包含。
> `arange(0, 1, 0.1)` 因为浮点误差可能给出 10 个或 11 个数，`linspace(0, 1, 10)` 永远是 10 个。
> 画图、算坐标的时候用 `linspace`，不会出错。

## 3. dtype：为什么类型这么重要

关键规则：**一个数组只能有一种类型**。混着放，numpy 会自动向上转型。

In [ ]:
a = np.array([1, 2, 3])
b = np.array([1, 2, 3.0])
c = np.array([True, False])
print('纯整数 :', a.dtype)
print('有小数 :', b.dtype)
print('布尔   :', c.dtype)

print('/2 之后:', (a / 2).dtype, '  <- 除法一定得到浮点，不会像 C 那样丢掉小数')
print('// 之后:', (a // 2).dtype)

small = np.array([1, 2, 3], dtype=np.int8)
print('int8 占内存:', small.nbytes, '字节（3 个元素）')
print('转成 float32:', a.astype(np.float32).dtype)

`int64` 每个元素 8 字节，`float32` 只有 4 字节。
图片、模型参数动辄几千万个数，**用 float32 而不是 float64，显存和内存直接省一半**——
这就是深度学习里到处能看到 `torch.float32` 的原因。

## 4. 形状：reshape 与 -1

数组的「形状」是一个元组，表示每一维有多长。

In [ ]:
grid = np.arange(1, 13)
print('原始     :', grid.shape)

m = grid.reshape(3, 4)
print('reshape 后:\n', m)
print('shape :', m.shape)
print('ndim  :', m.ndim, ' 维')
print('size  :', m.size, ' 个元素')
print('转置后 shape:', m.T.shape)

# -1 表示「这一维你自己算」，省得自己除
print('reshape(3, -1) 的 shape:', grid.reshape(3, -1).shape)
print('reshape(-1, 1) 的 shape:', grid.reshape(-1, 1).shape, '  <- 变成一列，后面很常用')

## 5. 索引与切片

二维数组用 `m[行, 列]` 一次给两个坐标，比 `m[行][列]` 更好也更高效。

In [ ]:
x = np.arange(10)
print('x          :', x)
print('x[2:5]     :', x[2:5], '  <- 含头不含尾')
print('x[::2]     :', x[::2], '  <- 隔一个取一个')
print('x[::-1]    :', x[::-1], '  <- 倒过来，最省事的反转')

m = np.arange(1, 13).reshape(3, 4)
print()
print('m =\n', m)
print('m[0, 1]  :', m[0, 1], '    <- 第 0 行第 1 列')
print('m[:, 1]  :', m[:, 1], ' <- 所有行的第 1 列')
print('m[1, :]  :', m[1, :], ' <- 第 1 行全部')
print('m[1:, 2:]:\n', m[1:, 2:], ' <- 行列同时切')

In [ ]:
# 花式索引：给一组下标，一次取多个不连续的位置
print('m[[0, 2], 1] :', m[[0, 2], 1], '  <- (0,1) 和 (2,1) 两个位置')

# 布尔索引：最重要的一种，写条件就完事
print('x[x % 2 == 0] :', x[x % 2 == 0])
print('x[x > 5]      :', x[x > 5])
print('x 大于 5 的个数:', (x > 5).sum(), '  <- True 当 1 相加，这招很好用')

# 两个条件要写括号
print('x 在 3~7 之间  :', x[(x >= 3) & (x <= 7)])

> **易错点**：多个条件必须用 `&` `|` `~`，不能用 `and` `or` `not`，而且每个条件都要套括号。
> `x >= 3 and x <= 7` 会报 `ValueError`——因为 Python 的 `and` 要求整个对象是布尔值，
> 而 `x >= 3` 给出的是一个布尔**数组**。

## 6. 视图 vs 副本：numpy 最容易踩的坑

**切片给的是「视图」，不是复制。** 你改切片，原数组跟着变。

In [ ]:
a = np.arange(10)
view = a[2:5]
view[0] = 999
print('改了切片之后，原数组 a 变成:', a)
print('  -> 切片是视图，共享同一块内存\n')

b = a[[2, 3, 4]]   # 花式索引
b[0] = -1
print('改了花式索引之后，a 还是:', a)
print('  -> 花式索引给的是副本，互不影响\n')

c = a[2:5].copy()  # 想要独立的就显式 copy
c[0] = 0
print('显式 copy 之后，a 还是:', a)

一句话记住：**切片（`a[2:5]`）给视图，花式索引和布尔索引（`a[[2,3]]`、`a[a>3]`）给副本。**

为什么要这么设计？因为视图不占额外内存。处理几个 G 的数据时，
`data[1000:2000]` 如果每次都复制一份，内存立刻爆掉。

这个坑后面会换个面貌再出现一次——pandas 3.0 默认开启 Copy-on-Write 之后，
链式赋值会**静默失效**（不报错，但值没变）。今天先埋个种子，第 2 课会专门收拾它。

## 7. 聚合与 axis（本节课的重点，慢一点）

`axis` 是初学者最容易混的概念。用一个 3 行 4 列的矩阵手动推一遍，比看十遍文档管用。

In [ ]:
m = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
])
print('m =\n', m)
print('shape:', m.shape, '  -> 第 0 维长 3（行），第 1 维长 4（列）\n')

print('m.sum()        =', m.sum())
print('m.sum(axis=0)  =', m.sum(axis=0), '  shape', m.sum(axis=0).shape)
print('m.sum(axis=1)  =', m.sum(axis=1), '      shape', m.sum(axis=1).shape)
print()
print('m.mean(axis=0) =', m.mean(axis=0))
print('m.max(axis=1)  =', m.max(axis=1))
print('m.std(axis=0)  =', m.std(axis=0).round(3))

### 三种理解方式，挑一个你能记住的

看上面的输出，`m.sum(axis=0)` 是 `[15, 18, 21, 24]`——正好是**每一列的和**（1+5+9=15）。

1. **「结果里没有第 i 维」**：`axis=0` 之后 shape 从 `(3, 4)` 变成 `(4,)`，第 0 维消失了。
2. **「沿着哪一维压扁」**：`axis=0` 是把行方向和起来、压成一列结果。
3. **口诀**：`axis=0` 竖着加（跨行），`axis=1` 横着加（跨列）。

第 1 种说法最严谨，也最通用——**三维、四维张量里同样成立**，
深度学习的 batch 维度就是靠它推出来的（`logits.sum(axis=1)` 这种写法你会看到很多次）。

## 8. 广播机制

广播（broadcasting）让形状不同的数组能直接运算，**不用写循环也不用复制数据**。

In [ ]:
m = np.arange(12).reshape(3, 4)
print('m =\n', m, '\n')

# 情况 1：标量，最简单
print('m + 100 =\n', m + 100)

# 情况 2：一维数组 (4,) 自动扩展成 (3, 4)，每行加同一组数
print('\nm + [1,2,3,4] =\n', m + np.array([1, 2, 3, 4]))

# 情况 3：列向量 (3,1) 与行向量 (1,4) 相加 -> (3,4)
print('\n形状 (3,1) + (1,4) 得到:', (np.ones((3, 1)) + np.ones((1, 4))).shape)

In [ ]:
# 广播规则：从右往左对齐维度，每一对要么相等，要么其中一个是 1
print('(3,4) 与 (4,)  -> 补成 (3,4) 与 (1,4) -> 兼容')
print('(3,4) 与 (3,)  -> 补成 (3,4) 与 (1,3) -> 从右往左第一对 4 和 3 不相等 -> 报错\n')

try:
    np.ones((3, 4)) + np.ones(3)
except ValueError as exc:
    print('报错信息:', exc)
    print('  -> 看到这种报错，先检查两个数组的 shape，八成就对上了')

### 广播到底有什么用

最常见的用途是**按列做标准化**：每列减掉自己那列的均值。
有了广播，一行就写完了，不需要循环、不需要转置。

In [ ]:
m = np.arange(12).reshape(3, 4).astype(float)

col_mean = m.mean(axis=0)
print('每列均值     :', col_mean.round(3))
print('它的 shape   :', col_mean.shape, ' -> 广播回去正好对上 (3,4)')

centered = m - col_mean
print('\n减完之后每列均值:', centered.mean(axis=0).round(10), ' <- 全是 0（10 个小数位）')

## 9. 小实战：自己实现一个标准化

z-score 标准化公式：`z = (x - 均值) / 标准差`。
用 numpy 写出来，你会发现它其实就是一句算术。

In [ ]:
rng = np.random.default_rng(0)
heights = rng.normal(170, 8, 1000)  # 1000 个人的身高，均值 170，标准差 8

z = (heights - heights.mean()) / heights.std()

print(f'原始数据   均值 {heights.mean():8.3f}   标准差 {heights.std():6.3f}')
print(f'标准化之后 均值 {z.mean():8.3f}   标准差 {z.std():6.3f}')
print()
print('min / max:', heights.min().round(2), '/', heights.max().round(2))
print('生成 10 个正态随机数:', rng.normal(0, 1, 10).round(3))

以后你会学到 `sklearn.preprocessing.StandardScaler`，它做的就是这件事：
记下训练集的均值和标准差，然后把训练集和测试集都这么处理一遍。

**先自己用 numpy 写一遍，再看 sklearn，那些「魔法」就都是算术了。**
这是我给整个课程定的基调：先懂原理，再用封装。

## 10. 练习（约 1 小时）

下面每格都是空的。先自己写，卡住超过 15 分钟再看提示（在最后一节）。

**做完不用发我截图，直接把每格的输出发我，我批改。**

### 练习 1（热身）

一行代码生成 1 到 100 之间所有 3 的倍数。

In [ ]:
# 你的代码
# 期望输出：开头是 3 6 9 12，最后一个是 99，个数是 33

### 练习 2（布尔索引）

把 `a` 里所有负数替换成 0。要求：**不许用 for 循环**，一行搞定。

In [ ]:
a = np.array([3, -1, 7, -5, 9, -2])

# 你的代码

# 期望输出: [3 0 7 0 9 0]

### 练习 3（axis）

造一个 4 行 5 列的随机整数矩阵（范围 0~50，随机种子用 42），然后分别求出：

- 每一列的最大值
- 每一行的平均值
- 整个矩阵的标准差

In [ ]:
# 你的代码

# 自检：每列最大值应该有 5 个数，每行平均值应该有 4 个数，标准差是一个数

### 练习 4（广播）

用广播生成 9×9 的乘法表（九九表），值就是行列相乘。
提示：一个 (9,1) 的列向量乘一个 (9,) 的行向量。

In [ ]:
# 你的代码

# 自检：shape 应该是 (9, 9)，且 [8, 8] 位置的值是 81

### 练习 5（模拟，这题最有趣）

模拟抛硬币：分别抛 10 次、100 次、10000 次，算出每次的正面比例。
看看比例是怎么慢慢靠近 0.5 的——这就是**大数定律**。

提示：`rng.integers(0, 2, size=n)` 生出 n 个 0/1，`.mean()` 就是正面比例。

In [ ]:
rng = np.random.default_rng(42)

# 你的代码

# 期望：抛 10 次的比例离 0.5 比较远，10000 次非常接近 0.5

### 练习 6（进阶，选做）

给定数组 `data`（20 行 3 列），把**每一列**做 z-score 标准化：
每列减掉该列均值、再除以该列标准差。要求只用广播，不用循环。

做完验证：结果的每列均值应该约等于 0，每列标准差约等于 1。

In [ ]:
rng = np.random.default_rng(1)
data = rng.normal(50, 10, (20, 3))

# 你的代码

# 验证：打印标准化后每列的均值（应为 0）和标准差（应为 1）

## 11. 总结卡片

抄到自己的笔记里，三句话：

1. **numpy 快的原因**：同类型 + 连续内存 + 循环在 C 里跑，这叫向量化。写代码时优先想「整体操作」而不是「逐元素循环」。
2. **`axis=i` 的含义**：结果里没有第 i 维。`axis=0` 竖着加，`axis=1` 横着加。
3. **视图 vs 副本**：切片给视图（会互相影响），花式索引和布尔索引给副本（互不影响）。

还有两个顺手记住的：

- `arange` 给步长、终点不含；`linspace` 给个数、终点包含。
- 多条件筛选用 `&` `|`，每个条件都要括号。

---

### 下一课预告

numpy 数组很厉害，但它有先天缺陷：**没有名字**。你只能说「第 3 列」，不能说「票价」这一列。

下一课 pandas 要做的事，就是给数组贴上标签——列名、行索引——然后就有了表格（DataFrame）。
在读数据之前，先想一个问题：一份 CSV 读到内存里，凭什么能变成带列名的表格？

## 12. 卡住时的提示（先别看）

**练习 1**：`np.arange` 的第一、二个参数是起点和终点，第三个是步长。起点想清楚从几开始，终点想清楚会不会被包含。

**练习 2**：你可以用布尔数组当「下标」去赋值，而不只是取值。先构造出「哪些位置是负数」的布尔数组，再拿它当下标。

**练习 3**：三个问题分别问的是「跨行还是跨列」。回想第 7 节的三种理解方式——结果里要消掉哪一维？标准差用 `std()`。

**练习 4**：关键在于把两个一维数组变成正确的形状：一个必须是列（多一维），另一个保持行。`reshape(9, 1)` 和 `reshape(1, 9)` 谁的形状是 `(9, 1)`？

**练习 5**：先生成 0/1 数组，再想「比例」在数学上就是「平均值」。三个规模写在同一个循环里会更清楚。

**练习 6**：第 8 节最后那个例子已经把「减均值」做完了。除以标准差是同样的道理——只要确认两个数组的形状能被广播就行。如果报形状错误，先打印两者的 `.shape` 看一看。